# Korea IPO historical validation audit

This notebook is the reproducible companion to `MASTER_PLAN.md`. It reads the local
SQLite database without changing it, excludes explicitly marked samples, separates
SPACs from operating companies, and treats D0/D1 returns as gross market outcomes.
It does **not** claim that `rules-v3-simple` is effective until point-in-time feature
snapshots and at least 30 complete score/outcome pairs exist.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.db import SessionLocal  # noqa: E402
from app.services.scoring_v3 import resolve_scoring_identity  # noqa: E402
from app.services.validation import assess_data_readiness, build_validation_dataset  # noqa: E402

session = SessionLocal()
model_version, rules_hash = resolve_scoring_identity()
readiness = assess_data_readiness(
    session,
    model_version=model_version,
    rules_hash=rules_hash,
    minimum_samples=30,
)
readiness.to_dict()

{'as_of_date': datetime.date(2026, 7, 19),
 'ipo_count': 28,
 'real_ipo_count': 27,
 'sample_ipo_count': 1,
 'listed_count': 26,
 'ticker_count': 26,
 'offer_price_count': 26,
 'demand_count': 0,
 'structure_count': 0,
 'listing_price_count': 26,
 'outcome_count': 26,
 'eligible_score_count': 0,
 'complete_pair_count': 0,
 'minimum_samples': 30,
 'selected_rules_hash': 'e3db68a1b566550c574515aeb5a9fa6a90d6493aac7fc5261983684fdc313d2c',
 'eligible_rules_hash_count': 0,
 'validation_ready': False,
 'blockers': ('과거 수요예측 피처 커버리지가 부족합니다.',
  '과거 상장일 유통물량 피처 커버리지가 부족합니다.',
  '누수 없는 사전 점수-D0/D1 완전 표본이 0건으로 최소 30건보다 적습니다.')}

## Data pipeline readiness

In [2]:
coverage = pd.DataFrame(
    [
        ("Listed by as-of date", readiness.listed_count),
        ("Ticker resolved", readiness.ticker_count),
        ("D0/D1 price", readiness.listing_price_count),
        ("Gross outcome", readiness.outcome_count),
        ("Historical demand feature", readiness.demand_count),
        ("Historical supply feature", readiness.structure_count),
        ("Eligible v3 score/outcome pair", readiness.complete_pair_count),
    ],
    columns=["stage", "count"],
)
coverage

,stage,count
0,Listed by as-of date,26
1,Ticker resolved,26
2,D0/D1 price,26
3,Gross outcome,26
4,Historical demand feature,0
5,Historical supply feature,0
6,Eligible v3 score/outcome pair,0


## Observed gross D0/D1 outcomes

In [3]:
segment_sql = text(
    "WITH ranked_complete AS (\n  SELECT\n    i.id AS ipo_id,\n    CASE WHEN instr(i.corp_name, '스팩') > 0\n         THEN 'SPAC' ELSE 'Operating company' END AS segment,\n    o.d0_open_return_pct,\n    o.d1_close_return_pct,\n    ROW_NUMBER() OVER (\n      PARTITION BY i.id\n      ORDER BY o.evaluated_at DESC, o.id DESC\n    ) AS outcome_rank\n  FROM outcome_evaluations o\n  JOIN ipos i ON i.id = o.ipo_id\n  WHERE i.is_sample = 0\n    AND o.d0_open_return_pct IS NOT NULL\n    AND o.d1_close_return_pct IS NOT NULL\n)\nSELECT\n  segment,\n  COUNT(*) AS n,\n  COUNT(DISTINCT ipo_id) AS unique_ipo_n,\n  ROUND(AVG(d0_open_return_pct), 2) AS avg_d0_open_return_pct,\n  ROUND(AVG(d1_close_return_pct), 2) AS avg_d1_close_return_pct,\n  ROUND(100.0 * SUM(d1_close_return_pct > 0) / COUNT(*), 1) AS d1_positive_pct\nFROM ranked_complete\nWHERE outcome_rank = 1\nGROUP BY segment\nORDER BY segment"  # noqa: E501
)
segment_outcomes = pd.read_sql_query(segment_sql, session.connection())
assert (segment_outcomes["n"] == segment_outcomes["unique_ipo_n"]).all()
segment_outcomes = segment_outcomes.drop(columns=["unique_ipo_n"])
segment_outcomes

,segment,n,avg_d0_open_return_pct,avg_d1_close_return_pct,d1_positive_pct
0,Operating company,18,162.19,102.61,72.2
1,SPAC,8,144.84,2.88,25.0


The table above is a small, date-concentrated descriptive baseline. It is not a
causal estimate and is not evidence that any score pillar predicts returns. D0-open
and D1-close are separate exit policies; fees, allocation probability, and cash
opportunity cost are excluded.

## Point-in-time v3 validation dataset

In [4]:
validation_rows = build_validation_dataset(
    session,
    model_version=model_version,
    rules_hash=rules_hash,
)
validation = pd.DataFrame([row.to_dict() for row in validation_rows])
print(f"eligible rows: {len(validation)}")
validation.head()

eligible rows: 0


""


## Integrity assertions

In [5]:
assert readiness.ipo_count == readiness.real_ipo_count + readiness.sample_ipo_count
assert readiness.ticker_count <= readiness.listed_count
assert readiness.listing_price_count <= readiness.listed_count
assert readiness.outcome_count <= readiness.listing_price_count
assert readiness.complete_pair_count <= readiness.eligible_score_count
assert readiness.complete_pair_count <= len(validation_rows)
assert readiness.selected_rules_hash == rules_hash
assert readiness.eligible_rules_hash_count <= 1
assert len(validation_rows) == readiness.complete_pair_count
assert readiness.validation_ready == (not readiness.blockers)

session.close()
print("All point-in-time readiness assertions passed.")

All point-in-time readiness assertions passed.
